# 01 — Ingestion & Chunking

Загрузка сырого корпуса, очистка, две стратегии чанкинга, запись в `data/chunks.jsonl`. Логика — в `scripts/build_index.py` (её же используют production-сборка и тесты).

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'scripts'))

from build_index import (
    load_documents, clean_text, chunk_recursive, chunk_section,
    build_chunks, write_chunks_jsonl,
)

RAW = str(ROOT / 'data' / 'raw')


## 1. Загрузка

In [ ]:
docs=load_documents(RAW); src,raw=docs[0]
print(len(docs),src,len(raw)); print(raw[:300])

## 2. Очистка

In [ ]:
clean=clean_text(raw); print(len(clean)); print(clean[:300])

## 3. Две стратегии чанкинга (recursive обязателен + section)

In [ ]:
rec=chunk_recursive(src,clean,800,150); sec=chunk_section(src,clean)
print('recursive:',len(rec),'| section:',len(sec))
print(sec[0].section_title,'->',sec[0].text[:100])

## 4. Метаданные чанка

In [ ]:
print(sec[2].model_dump())

## 5. Персист в JSONL (в проде — `scripts/build_index.py`)

In [ ]:
n=write_chunks_jsonl(build_chunks(RAW,strategy='section'),str(ROOT / 'data' / 'chunks.jsonl')); print('чанков:',n)